Use the rexomni environment

In [1]:
import torch
from PIL import Image
from pathlib import Path
import json
import os, sys
import time

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(repo_root)
sys.path.append(os.path.join(repo_root, "src"))

from annotation_methods.io_utils import write_coco_output, retrieve_image_batches

from rex_omni import RexOmniVisualize, RexOmniWrapper


In [2]:
# ---- Reusable constants ----
DATA_ROOT = Path("../../Data")

RESULTS_PATH = Path("../../Results/Experiment_1")

DATASETS = ["apples", "tomatoes"]
DATASET_DICT = {
    "apples": ["good apple", "bad apple"],
    "tomatoes": ["tomato"],
}

In [ ]:
def initiate_rex_model(backend:str):
    if backend == "transformers":
        # Initialize the wrapper (model loads internally)
        rex_model = RexOmniWrapper(
            model_path="IDEA-Research/Rex-Omni",   # HF repo or local path
            backend="transformers",                # or "vllm" for high-throughput inference
            max_tokens=2048,
            temperature=0.0,
            top_p=0.05,
            top_k=1,
            repetition_penalty=1.05,
        )
    if backend == "vllm":
        # Initialize the wrapper (model loads internally)
        rex_model = RexOmniWrapper(
            model_path="IDEA-Research/Rex-Omni",   # HF repo or local path
            backend="vllm",
            max_tokens=2048,
            temperature=0.0,
            top_p=0.05,
            top_k=1,
            repetition_penalty=1.05,
        )        
    return rex_model

In [ ]:
def run_rex_inference_to_coco(
    dataset: str,
    rex_backend: str,
    data_root: Path,
    output_path: Path,
    categories: list,
    batch_size: int = 4,
    sample_size: int | None = None,
    model_name: str = "rexomni",
    warmup_steps: int = 1, 
):
    """
    Run REX detection inference on a test split and write COCO-format JSON.

    Parameters
    ----------
    dataset : str
        Dataset name, used to look up categories and paths.
    rex_backend : str
        Backend identifier for initiate_rex_model.
    data_root : Path
        Root path of all datasets (parent of `<dataset>/images/test`).
    output_path : Path
        Directory where the COCO JSON and logs will be written.
    categories : list
        List of category names.
    batch_size : int, optional
        Batch size for retrieve_image_batches.
    sample_size : int | None, optional
        Sample size for retrieve_image_batches (pass None for all images).
    model_name : str, optional
        Name to pass to write_coco_output. (is updated with rex_backend)
    warmup_steps : int , optional
        Pass how many batches should be processed to warmup the kernel (default = 1)

    Returns
    -------
    out_file : str or Path
        Path returned by write_coco_output.
    """

    # Resolve paths and categories
    images_folder = data_root / dataset / "images" / "test"

    images = []
    annotations = []

    img_id = 1
    ann_id = 1
    num_pred_boxes = 0
    total_inf_time = 0.0
    first_batch = True

    model_name = f'{model_name}_{rex_backend}'

    # Load the model
    rex_model = initiate_rex_model(rex_backend)

    warmup_steps = warmup_steps

    # Run inference
    for batch_images, names in retrieve_image_batches(
        images_folder, batch_size=batch_size, sample_size=sample_size
    ):
        if first_batch:
            # Warmup on first batch (not timed, no results stored)
            for _ in range(warmup_steps):
                _ = rex_model.inference(
                    images=batch_images, task="detection", categories=categories
                )
            first_batch = False

        # Timed inference (first batch is reprocessed here and results are stored)
        start = time.perf_counter()
        results = rex_model.inference(
            images=batch_images, task="detection", categories=categories
        )
        end = time.perf_counter()
        total_inference_time_s += (end - start)

        # COCO conversion
        for name, res, im in zip(names, results, batch_images):
            # Skip if inference failed
            if not res.get("success", False):
                continue

            # Original image size: (width, height)
            w, h = res["image_size"]

            # Register image entry
            current_img_id = img_id
            images.append(
                {
                    "id": current_img_id,
                    "file_name": f"images/test/{name}",
                }
            )
            img_id += 1

            # Extract predictions for this image
            preds = res.get("extracted_predictions", {})
            for label, objs in preds.items():
                if label not in categories:
                    continue
                category_id = categories.index(label)

                for obj in objs:
                    if obj.get("type") != "box":
                        continue

                    x0, y0, x1, y1 = obj["coords"]

                    # Clamp coordinates to image bounds
                    x0 = max(0.0, min(float(x0), float(w)))
                    x1 = max(0.0, min(float(x1), float(w)))
                    y0 = max(0.0, min(float(y0), float(h)))
                    y1 = max(0.0, min(float(y1), float(h)))

                    # Convert [x0, y0, x1, y1] -> [x, y, width, height]
                    bbox_w = max(0.0, x1 - x0)
                    bbox_h = max(0.0, y1 - y0)
                    area = bbox_w * bbox_h

                    # Skip degenerate boxes
                    if bbox_w <= 0 or bbox_h <= 0:
                        continue

                    annotations.append(
                        {
                            "id": ann_id,
                            "image_id": current_img_id,
                            "category_id": category_id,
                            "bbox": [x0, y0, bbox_w, bbox_h],
                            "area": area,
                            "iscrowd": 0,
                        }
                    )
                    ann_id += 1
                    num_pred_boxes += 1

    # ---------------------------------------------------------------------
    # Write COCO JSON and return path
    # ---------------------------------------------------------------------
    num_images = len(images)

    out_file = write_coco_output(
        images_folder=str(images_folder),
        model_name=model_name,
        categories_list=categories,
        images=images,
        annotations=annotations,
        num_images=num_images,
        output_path=output_path,
        num_initial_bbox=0,  # zero-shot has no initial boxes
        num_pred_boxes=num_pred_boxes,
        total_inference_time_s=total_inference_time_s,
        machine_training_time_s=0.0,  # zero-shot requires no training time
    )

    print(f"Wrote COCO-format JSON to {out_file}")
    return out_file


In [ ]:
dataset = "apples"
rex_backend = "transformers"
# rex_backend = "vllm"
categories = DATASET_DICT.get(dataset)

run_rex_inference_to_coco(dataset=dataset, 
                          rex_backend=rex_backend,
                          data_root=DATA_ROOT,
                          output_path=RESULTS_PATH,
                          categories=categories,
                          batch_size=6,
                          sample_size=None)

Initializing transformers backend...


/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.26it/s]


Found 31 image files


/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.05` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 12.11 GiB. GPU 0 has a total capacity of 22.06 GiB of which 6.23 GiB is free. Including non-PyTorch memory, this process has 15.82 GiB memory in use. Of the allocated memory 15.10 GiB is allocated by PyTorch, and 430.82 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

: 